# Regression: finite point predictions and pinned reuse

Default execution uses a synthetic HTTP fixture, not an FM or a quality benchmark. Explicit live execution requires the separately qualified TabICLv2 regression task and matching ready worker. This example uses the frozen planted fixture: seed42, 128 training rows, four numeric features and 32 query rows. It never substitutes classification or another model. Read README.md before live use.

In [ ]:
from support import notebook_client, prepare_regression
import numpy as np
client = notebook_client()
columns, query, predictor = prepare_regression(client)
assert predictor["task"] == "regression"
assert predictor["model_id"] == "tabicl-v2"
assert predictor.get("classes") is None
pinned_version = predictor["model_version"]

Preparation binds the training table, seed, configuration and task version. Point predictions are in target units; these are not class probabilities, calibrated intervals or conformal coverage. This notebook checks output semantics rather than imposing a scientific accuracy threshold.

In [ ]:
result = client.predict(predictor["id"], columns=columns, rows=query.tolist())
points = np.asarray(result["predictions"], dtype=float)
assert points.shape == (32,)
assert np.isfinite(points).all()
assert result.get("classes") is None
assert result.get("probabilities") is None
assert result["model_version"] == pinned_version

Named columns may arrive in a different order. The service aligns the same features without refitting. Reopening reuses the persisted predictor identity and model version; it does not force a process restart or prove cold restoration.

In [ ]:
reordered = client.predict(predictor["id"], columns=columns[::-1], rows=query[:, ::-1].tolist())
assert np.allclose(reordered["predictions"], points, atol=1e-5, rtol=1e-5)
reopened = client.predictor(predictor["id"])
repeat = client.predict(reopened["id"], columns=columns, rows=query.tolist())
assert np.allclose(repeat["predictions"], points, atol=1e-5, rtol=1e-5)
assert repeat["model_version"] == pinned_version
assert reopened["task"] == "regression"
client.close()

The accepted synthetic dataset, predictor and normal server job/results remain retained. No deletion or saved prediction output is part of this example. Other selected models, model swapping, native uncertainty, large batch workflows and causal/ergodic examples require their own actual qualification and contracts.